In [1]:
from pathlib import Path
import os
import shutil


In [2]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

In [3]:
project_dir = Path(os.getcwd()).parent.parent
install_dir = project_dir / "install"
log_dir = project_dir / "logs" / "dlio"
data_dir = Path("/p/lustre5/haridev/dlio_demo")
output_dir = project_dir / "output" / "dlio"
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /usr/WS2/haridev/dftracer-demo/install
Log Directory: /usr/WS2/haridev/dftracer-demo/logs/dlio
Data Directory: /p/lustre5/haridev/dlio_demo
Output Directory: /usr/WS2/haridev/dftracer-demo/output/dlio


In [4]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


In [5]:
import os

import importlib.util

spec = importlib.util.find_spec("dftracer")
if spec and spec.origin:
    print("dftracer module path:", spec.origin)
    dftracer_folder = os.path.dirname(spec.origin)
    print("dftracer folder:", dftracer_folder)
else:
    print("dftracer module not found.")

dftracer module path: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer/__init__.py
dftracer folder: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer


[Documentation](https://dftracer.readthedocs.io/en/latest/api.html)

In [6]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1

# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

export DFTRACER_BIND_SIGNALS=1

export DFTRACER_TRACE_COMPRESSION=1

echo "Activating environment"
source {project_dir}/demo/dlio/setup_env.sh {install_dir} 2> /dev/null


echo "Running DLIO with DFTracer"
flux run -n 2 -o fastload -q pbatch {install_dir}/bin/dlio_benchmark workload=unet3d_a100 ++workload.workflow.generate_data=True hydra.run.dir={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.dataset.num_files_train=32 ++workload.dataset.record_length_bytes=1048576 ++workload.dataset.data_folder={data_dir}/unet3d_a100/data ++workload.checkpoint.checkpoint_folder={data_dir}/unet3d_a100/checkpoint ++workload.train.epochs=1 > {output_dir}/log.txt 2> {output_dir}/error.txt || true
echo "Finished running DLIO with DFTracer"

Configuring DFTracer
Activating environment
Running DLIO with DFTracer


In [7]:
import glob

pfw_files = glob.glob(str(output_dir/ "unet3d_a100" / "*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)


Found .pfw.gz files:
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0-of-2.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-1-of-2.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-d38d5f8086171392-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0dd95c5f4dc13d6c-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-ef53986f03c4f9d1-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-a575de18680e4a01-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-e0c09c229f2ce945-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-a0b2b3336f6beec9-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-f00e34187ca4d020-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-f25c0ffabb9c4df1-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-7f069feadfc94119-app.pfw.gz
/usr/WS2/haridev/dftracer-de

In [8]:
%%pybash
{install_dir}/bin/dftracer_split -n unet3d -f -d {output_dir}/unet3d_a100 -o {output_dir}/unet3d_a100/compact

Arguments:
  App name: unet3d
  Override: 1
  Data dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100
  Output dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact
  Chunk size: 1024


12/08/2025 14:01:46 Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
12/08/2025 14:01:46 Found zq executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zq
12/08/2025 14:01:46 sqlite3 exists
12/08/2025 14:01:46  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100: 18
12/08/2025 14:01:46  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
12/08/2025 14:01:46  Removing existing indices as override is passed.
12/08/2025 14:01:46  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0dd95c5f4dc13d6c-app.pfw.gz
12/08/2025 14:01:46  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-22032bff4c706d49-app.pfw.gz
12/08/2025 14:01:46  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace

rm: cannot remove '/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/counting.bak': No such file or directory


12/08/2025 14:01:54 Completed collecting size .03125095 17 of 18                                
12/08/2025 14:01:54 Finished collecting data from 18 tasks
12/08/2025 14:01:54 Scheduled chunks: 0
12/08/2025 14:01:54 Total chunks: 1
12/08/2025 14:01:54 Start processing chunks
12/08/2025 14:04:06 Chunk 1 out of 1 done with size 8.44830981 MB, path = /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact/unet3d-1.pfw                               
12/08/2025 14:04:06 All chunks processed
12/08/2025 14:04:06 re-index split files
12/08/2025 14:04:06  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact: 1
12/08/2025 14:04:06  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
12/08/2025 14:04:06  Removing existing indices as override is passed.
12/08/2025 14:04:06  Compressing file unet3d-1
12/08/2025 14:04:06  Created index for compressed file /usr/WS2/haridev/dftracer-demo/ou

12/08/2025 14:04:15 Error: Original lines count 36184 does not match split lines count 36176. Please check the file. 
12/08/2025 14:04:15 Done re-index of split files


In [9]:
!gzip -dc {output_dir}/unet3d_a100/compact/*.pfw.gz | (head -n 10; echo "..."; tail -n 5)

[
{"id":1,"name":"HH","cat":"dftracer","pid":67335,"tid":67335,"ph":"M","args":{"hhash":"a32d1bfb90ba693f","name":"tuolumne1147","value":"a32d1bfb90ba693f"}}
{"id":2,"name":"thread_name","cat":"dftracer","pid":67335,"tid":67335,"ph":"M","args":{"hhash":"a32d1bfb90ba693f","name":"67335","value":"thread_name"}}
{"id":3,"name":"SH","cat":"dftracer","pid":67335,"tid":67335,"ph":"M","args":{"hhash":"a32d1bfb90ba693f","name":"/usr/WS2/haridev/dftracer-demo/install/bin/python;/usr/WS2/haridev/dftracer-demo/install/bin/dlio_benchmark;workload=unet3d_a100;++workload.workflow.generate_data=True;hydra.run.dir=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.dataset.num_files_train=32;++workload.dataset.record_length_bytes=1048576;++workload.dataset.data_folder=/p/lustre5/haridev/dlio_demo/unet3d_a100/data;++workload.che

In [10]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/unet3d_a100/compact/",
        f"analyzer/preset=dlio",
        
    ]
)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34321 instead
  warnings.warn(


In [11]:
dfa.client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:34321/status,
Dashboard: http://127.0.0.1:34321/status,Workers: 12
Total threads: 96,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41843,Workers: 12
Dashboard: http://127.0.0.1:34321/status,Total threads: 96
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34063,Total threads: 8
Dashboard: http://127.0.0.1:33465/status,Memory: 0 B
Nanny: tcp://127.0.0.1:34769,


In [12]:
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:112: RuntimeWarning: invalid value encountered in scalar divide
  ops=float('nan') if pd.isna(time) else float(count / time),
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:113: RuntimeWarning: invalid value encountered in scalar divide
  bandwidth=float('nan') if pd.isna(time) or pd.isna(size) else float(size / time),


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                                    ┃ Unit            ┃             Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                                  │ seconds         │            43.167 │
│ Total Count                                                               │ count           │            35,859 │
│ Total Files                                                               │ count           │                 6 │
│ Total Nodes                                                               │ count           │                 0 │
│ Total Processes                                                           │ count           │                10 │
│ App Count                                                                 │ count           │                 2 │
│ Training Count                                                            │ count           │                 2 │
│ Compute Count                                                             │ count           │                 4 │
│ Fetch Data Count                                                          │ count           │                34 │
│ Data Loader Count                                                         │ count           │                36 │
│ Data Loader Fork Count                                                    │ count           │                24 │
│ Reader Count                                                              │ count           │               112 │
│ Reader POSIX (Lustre) Count                                               │ count           │            35,617 │
│ Reader POSIX (Lustre) Size                                                │ MB              │         15821.551 │
│ Reader POSIX (Lustre) Bandwidth                                           │ MB/s            │          1865.861 │
│ Reader POSIX (Lustre) Avg Transfer Size                                   │ MB              │             0.444 │
│ Checkpoint Count                                                          │ count           │                 1 │
│ Checkpoint POSIX (Lustre) Count                                           │ count           │                 3 │
│ Other POSIX Count                                                         │ count           │                24 │
└───────────────────────────────────────────────────────────────────────────┴─────────────────┴───────────────────┘
                                          Layer Breakdown (w/ overlap %)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Layer                       ┃       Time (s) ┃            Ops ┃   Ops/sec ┃        Size (MB) ┃ Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ App                         │  16.858 (----) │       2 (----) │     0.119 │                - │                - │
│ Training                    │  16.788 (----) │       2 (----) │     0.119 │                - │                - │
│ Compute                     │   2.545 (----) │       4 (----) │     1.572 │                - │                - │
│ Fetch Data                  │  12.110 (  0%) │      34 (  0%) │     2.808 │                - │                - │
│ Data Loader                 │  32.829 (  2%) │      36 ( 22%) │     1.097 │                - │                - │
│ Data Loader Fork            │   0.185 (  0%) │      24 (  0%) │   129.734 │                - │                - │
│ Reader                      │  31.815 (  2%) │     112